In [ ]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
import os
import importlib
import time
import itertools
import importlib

from kfirlib.comparisons import *
from kfirlib import util
from kfirlib import llm, prompts, pdfs
from importlib import reload

In [ ]:
reload(util)
reload(pdfs)

In [ ]:
DATA_DIR = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\tables'

In [ ]:
df = pd.read_excel(os.path.join(DATA_DIR, 'compare_4.xlsx'))
df.head()

In [ ]:
pdf_paths = df.pdf_path[df.pdf_path.notna()].tolist()
len(pdf_paths), len(set(pdf_paths))

In [ ]:
gemini_res = pdfs.evaluate(pdf_paths, model=llm.GEMINI_3_FLASH, criteria=prompts.PAUL_CRITERIA)
gemini_res.head()

In [ ]:
#all_gemini = pdfs.evaluate(pdf_rest_paths, model=llm.GEMINI_3_FLASH, criteria=prompts.JUMALAH_CRITERIA)
#all_gemini.decision.sum()

In [ ]:
gemini_res.head()

In [ ]:
len(gemini_res)

In [ ]:
#all_gemini.to_excel(r"C:\Users\israe\OneDrive\Documents\post doc\kfir\Jumalah\tables\1481_gemini_full_text.xlsx")
gemini_res.to_excel(r"C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\tables\307_gemini_full_text.xlsx")

In [ ]:
for fn in gemini_res[gemini_res.decision.isna()].filename:
    print(fn)

In [ ]:
missing_filenames = [os.path.basename(fn).removesuffix('**') for fn in gemini_res[gemini_res.decision.isna()].filename]
missing_filenames

In [ ]:
missing_filenames = set(missing_filenames)

In [ ]:
all_pdfs = util.list_files(r'C:\Users\israe\OneDrive\Documents\post doc\kfir\HPV\sophie\Paul_and_Sophie', '.pdf')
found_pdfs = [path for path in all_pdfs if os.path.basename(path) in missing_filenames]
len(missing_filenames), len(found_pdfs)

In [ ]:
found_pdfs

In [ ]:
extra_res = pdfs.evaluate(found_pdfs, model=llm.GEMINI_3_FLASH, criteria=prompts.PAUL_CRITERIA)
extra_res

In [ ]:
all_gemini_res = pd.concat([
    gemini_res[gemini_res.decision.notna()],
    extra_res,
], axis=0)
all_gemini_res.to_excel(r"C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\tables\307_gemini_full_text.xlsx", index=False)

In [ ]:
joined = pd.merge(df, all_gemini_res, left_on='pdf_path', right_on='filename', how='left')
joined.head()

In [ ]:
joined.columns

In [ ]:
joined.drop(columns=['filename_y'], inplace=True)
joined.rename(columns={'filename_x': 'filename', 'reasoning_x': 'reasoning', 'reasoning_y': 'reasoning_full_text', 'decision': 'decision_full_text'},
              inplace=True)

In [ ]:
len(joined)

In [ ]:
joined.to_excel(os.path.join(DATA_DIR, 'compare_4_with_full.xlsx'), index=False)